<a href="https://colab.research.google.com/github/allengr220/cognitive-ledger/blob/main/longevity/notebooks/physiorisk_canonical_frozen_baseline_notebook_levinenocrp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## PhysioRisk — Benchmark-Aligned Frozen Baseline Notebook (Levine-noCRP Variant, Locked)

### Overview

This notebook builds the **benchmark-aligned PhysioRisk score set** for the Levine-noCRP comparison pipeline.

The PhysioRisk framework decomposes mortality risk into:

$$
\text{TotalRisk} = \text{BaselineRisk} + \text{PhysioRisk}
$$

- **BaselineRisk** captures age/sex mortality hazard and is fit upstream
- **PhysioRisk** captures age-orthogonal physiological variation derived from residualized biomarkers

This notebook does **not** refit baseline. It consumes frozen baseline-scored train/test inputs and produces a **benchmark-ready scored export** for downstream comparison against Levine-style no-CRP models.

---

### Purpose

Using the benchmark-aligned NHANES cohort, this notebook:

1. loads train/test datasets that already contain `BaselineRisk`
2. applies train-only biomarker preprocessing
3. residualizes biomarkers against spline(age) + sex using **training data only**
4. constructs fixed physiological axes
5. computes `PhysioRisk_2axis`
6. combines physiology with frozen baseline to compute `TotalRisk_2axis`
7. exports train/test scored datasets containing:
   - survival variables
   - demographic variables
   - benchmark biomarker columns
   - frozen `BaselineRisk`
   - axis scores
   - final PhysioRisk and TotalRisk scores

---

### BaselineRisk (Frozen Upstream)

- Model: Cox proportional hazards
- Formula: `cr(RIDAGEYR, df=4) + C(RIAGENDR)`
- Fit: **training split only**, in the upstream baseline builder
- Application: applied unchanged to both train and test sets
- Penalization: `penalizer = 0.01`, `l1_ratio = 0.0`

**Invariant:**  
No baseline refitting occurs downstream of the baseline builder stage.

---

### Cohort Definition

- Dataset: NHANES 1999–2016 mortality-linked cohort
- Cohort type: **benchmark-aligned Levine-noCRP subset**
- Split: temporal
  - Train: `cycle_start_year <= 2010`
  - Test: `cycle_start_year >= 2011`

This cohort is restricted to support fair downstream comparison across:
- PhysioRisk benchmark variant
- TotalRisk benchmark variant
- Levine-style no-CRP comparator models

---

### Biomarker Processing

#### PhysioRisk modeling biomarkers
The primary physiology model uses the reduced benchmark-aligned biomarker subset needed for the fixed 2-axis specification.

- Missingness:
  - median imputation using **training-derived values**

- Residualization:
  - Model: linear regression
  - Covariates: `cr(RIDAGEYR, df=4) + C(RIAGENDR)`
  - Fit: **training data only**
  - Output: standardized residuals (`resid_z`)

**Invariant:**  
BaselineRisk is not used as a covariate in biomarker residualization or PhysioRisk estimation.

---

### Benchmark Export Columns

In addition to the PhysioRisk model outputs, exported train/test artifacts preserve the **Levine-noCRP benchmark biomarker fields** required for downstream comparator construction.

These exports are intended to be **self-sufficient benchmark artifacts**, not just minimal PhysioRisk score files.

---

### Axis Construction

All axis components are:

- residualized against spline(age) + sex
- sign-aligned so higher = worse
- standardized on the training set before aggregation

---

### PhysioRisk (Primary Benchmark Variant)

**Structure: 2-axis model**

Axes:
- HematologicStress
- RenalHepaticTurnover

Axis construction:
- each axis is the simple mean of its constituent residualized biomarkers

PhysioRisk is defined as:

$$
\text{PhysioRisk}_{2\text{-axis}} =
\frac{\text{HematologicStress} + \text{RenalHepaticTurnover}}{2}
$$

PhysioRisk is standardized on the training set prior to evaluation and export.

---

### TotalRisk

$$
\text{TotalRisk}_{2\text{-axis}} =
\text{BaselineRisk} + \text{PhysioRisk}_{2\text{-axis}}
$$

---

### Evaluation Outputs

Primary downstream evaluation uses:

- C-index
- hazard ratio per 1 SD
- correlation with chronological age

These metrics are evaluated separately on train and temporal test cohorts.

---

### Export Outputs

This notebook exports benchmark-ready train/test scored artifacts containing:

- identifier column
- survival time and event columns
- age and sex
- `cycle_start_year` where available
- Levine-noCRP benchmark biomarker columns
- `BaselineRisk`
- `HematologicStress`
- `RenalHepaticTurnover`
- `PhysioRisk_2axis`
- `TotalRisk_2axis`

Optional exports may also include:
- evaluation summary CSV
- metadata describing the locked specification

---

### Architectural Invariants

1. BaselineRisk is fit once upstream and then frozen
2. Residualization is fit on training data only
3. No test information is used during preprocessing or scoring
4. BaselineRisk is not reused as a covariate in PhysioRisk estimation
5. Axis definitions are fixed prior to evaluation
6. All benchmark comparisons use identical cohorts and temporal splits
7. Exported artifacts must preserve all columns required for downstream comparator construction

---

### Interpretation

This notebook implements a **benchmark-aligned PhysioRisk variant**, not the broader canonical EXTENDED_16 configuration.

Any performance differences relative to the canonical PhysioRisk notebook should be interpreted as arising from:
- reduced biomarker panel
- cohort alignment
- benchmark comparability

and **not** from changes in baseline architecture.

---

### Final Model

The primary benchmark model produced by this notebook is:

> **BaselineRisk + 2-axis PhysioRisk (hematologic + renal/hepatic) on the Levine-noCRP aligned cohort**

This specification is locked for benchmark analyses unless explicitly versioned.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install lifelines

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 409.1/409.1 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.3/117.3 kB 8.7 MB/s eta 0:00:00
  Created wheel for autograd-gamma: filename=autograd_gamma-0.5.0-py3-none-any.whl size=4030 sha256=2f5854c02945a20d835f84691d3c74345eb6f6e1eb6deaf402f21c528421a217
  Stored in directory: /root/.cache/pip/wheels/50/37/21/0a719b9d89c635e89ff24bd93b862882ad675279552013b2fb
Successfully built autograd-gamma


In [3]:
# Core imports

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression

from lifelines import CoxPHFitter
from lifelines.utils import concordance_index

from patsy import dmatrix, build_design_matrices


## Configuration

Set the train/test parquet paths to the outputs of the upstream baseline builder.

These files must already contain `BaselineRisk`.


In [4]:
# ---------------------------
# User configuration
# ---------------------------

TRAIN_PATH = Path("/content/drive/MyDrive/LRE/data/pub/baseline/levine_benchmark/physiorisk_v1_frozen_baseline_train_levinenocrp.parquet")
TEST_PATH  = Path("/content/drive/MyDrive/LRE/data/pub/baseline/levine_benchmark/physiorisk_v1_frozen_baseline_test_levinenocrp.parquet")

ID_COL = "SEQN"
TIME_COL = "PERMTH_INT"
EVENT_COL = "MORTSTAT"
AGE_COL = "RIDAGEYR"
SEX_COL = "RIAGENDR"
BASELINE_COL = "BaselineRisk"

# Residualization biomarker set (canonical 2-axis v1)
BIOMARKER_COLS = [
    "LBXSAL",
    "LBXSCR",
    "LBXSAPSI",
    "LBXWBCSI",
    "LBXLYPCT",
    "LBXMCV",
    "LBXRDW",
]

# Canonical 2-axis grouping
AXIS_GROUPS_V1 = {
    "HematologicStress": [
        "LBXWBCSI_resid_z",
        "LBXLYPCT_resid_z",
        "LBXMCV_resid_z",
        "LBXRDW_resid_z",
    ],
    "RenalHepaticTurnover": [
        "LBXSCR_resid_z",
        "LBXSAL_resid_z",
        "LBXSAPSI_resid_z",
    ],
}

# Sign convention: higher = worse
SIGN_FLIPS_V1 = {
    "LBXLYPCT_resid_z": -1.0,
    "LBXSAL_resid_z": -1.0,
}

PHYSIORISK_PENALIZER = 0.01


## Load frozen baseline-scored train/test data


In [5]:
train_df = pd.read_parquet(TRAIN_PATH).copy()
test_df  = pd.read_parquet(TEST_PATH).copy()

print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)

required_cols = [ID_COL, TIME_COL, EVENT_COL, AGE_COL, SEX_COL, BASELINE_COL] + BIOMARKER_COLS
missing_train = [c for c in required_cols if c not in train_df.columns]
missing_test  = [c for c in required_cols if c not in test_df.columns]

if missing_train:
    raise ValueError(f"Missing required columns in train_df: {missing_train}")
if missing_test:
    raise ValueError(f"Missing required columns in test_df: {missing_test}")

print("BaselineRisk present in both train/test.")


Train shape: (13090, 16)
Test shape : (6739, 16)
BaselineRisk present in both train/test.


## Guardrail checks


In [6]:
assert BASELINE_COL in train_df.columns and BASELINE_COL in test_df.columns, "BaselineRisk must exist in input files."
print("Guardrail check passed: notebook is using precomputed BaselineRisk inputs.")


Guardrail check passed: notebook is using precomputed BaselineRisk inputs.


## Train-only residualization utilities

Biomarkers are residualized against age and sex using **train-only** fitted transforms.
This notebook deliberately does not use BaselineRisk in residualization.


In [7]:
# Spline-based train-only residualization
# Uses the same age representation family as the frozen baseline builder:
# cr(RIDAGEYR, df=4) + RIAGENDR

def make_resid_design_matrix(df, age_col, sex_col, design_info=None):
    base = df[[age_col, sex_col]].copy()

    if design_info is None:
        X = dmatrix(
            f"cr({age_col}, df=4) + {sex_col}",
            base,
            return_type="dataframe",
        )
        return X, X.design_info
    else:
        X = build_design_matrices([design_info], base)[0]
        X = pd.DataFrame(
            X,
            index=base.index,
            columns=design_info.column_names,
        )
        return X, design_info


def fit_residualizers(train_df, biomarker_cols, age_col, sex_col):
    models = {}

    X_train, resid_design_info = make_resid_design_matrix(
        train_df,
        age_col=age_col,
        sex_col=sex_col,
        design_info=None,
    )

    for col in biomarker_cols:
        y = train_df[[col]].copy()

        imp_y = SimpleImputer(strategy="median")
        y_train = imp_y.fit_transform(y).ravel()

        lm = LinearRegression()
        lm.fit(X_train, y_train)

        train_pred = lm.predict(X_train)
        train_resid = y_train - train_pred

        resid_mean = float(np.mean(train_resid))
        resid_std = float(np.std(train_resid))
        if resid_std == 0:
            resid_std = 1.0

        models[col] = {
            "imp_y": imp_y,
            "lm": lm,
            "resid_mean": resid_mean,
            "resid_std": resid_std,
        }

    return {
        "design_info": resid_design_info,
        "models": models,
    }


def apply_residualizers(df, biomarker_cols, age_col, sex_col, resid_bundle):
    X, _ = make_resid_design_matrix(
        df,
        age_col=age_col,
        sex_col=sex_col,
        design_info=resid_bundle["design_info"],
    )

    out = pd.DataFrame(index=df.index)

    for col in biomarker_cols:
        bundle = resid_bundle["models"][col]
        y = bundle["imp_y"].transform(df[[col]]).ravel()

        pred = bundle["lm"].predict(X)
        resid = y - pred
        resid_z = (resid - bundle["resid_mean"]) / (bundle["resid_std"] + 1e-8)

        out[f"{col}_resid_z"] = resid_z

    return out

In [8]:
resid_bundle = fit_residualizers(
    train_df=train_df,
    biomarker_cols=BIOMARKER_COLS,
    age_col=AGE_COL,
    sex_col=SEX_COL,
)

resid_train = apply_residualizers(
    df=train_df,
    biomarker_cols=BIOMARKER_COLS,
    age_col=AGE_COL,
    sex_col=SEX_COL,
    resid_bundle=resid_bundle,
)

resid_test = apply_residualizers(
    df=test_df,
    biomarker_cols=BIOMARKER_COLS,
    age_col=AGE_COL,
    sex_col=SEX_COL,
    resid_bundle=resid_bundle,
)

print(resid_train.shape, resid_test.shape)
resid_train.head()

(13090, 7) (6739, 7)


,LBXSAL_resid_z,LBXSCR_resid_z,LBXSAPSI_resid_z,LBXWBCSI_resid_z,LBXLYPCT_resid_z,LBXMCV_resid_z,LBXRDW_resid_z
0,0.624133,-0.500832,-0.567833,0.380679,-0.716973,-0.696483,0.516762
1,0.477418,-0.437222,-0.345388,-0.333960,0.838292,-0.979459,0.341185
2,1.167727,-0.327134,0.030232,-0.803805,1.858741,-0.407594,1.077010
3,-0.396440,-0.445865,0.595257,-0.963430,1.282646,0.437730,0.883348
4,1.016480,-0.449694,-0.244335,1.482533,-0.764605,-1.136175,0.837646


## Apply sign conventions


In [9]:
def apply_signs(df, sign_flips):
    out = df.copy()
    for col, s in sign_flips.items():
        if col in out.columns:
            out[col] = out[col] * float(s)
    return out

resid_train_signed = apply_signs(resid_train, SIGN_FLIPS_V1)
resid_test_signed  = apply_signs(resid_test, SIGN_FLIPS_V1)


## Build axis scores


In [10]:
def build_axis_scores_from_groups(resid_df, axis_groups):
    out = pd.DataFrame(index=resid_df.index)
    for axis_name, cols in axis_groups.items():
        use_cols = [c for c in cols if c in resid_df.columns]
        if len(use_cols) == 0:
            raise ValueError(f"No usable columns found for axis {axis_name}")
        out[axis_name] = resid_df[use_cols].mean(axis=1)
    return out

axis_train = build_axis_scores_from_groups(resid_train_signed, AXIS_GROUPS_V1)
axis_test  = build_axis_scores_from_groups(resid_test_signed, AXIS_GROUPS_V1)

axis_train.describe().T


,count,mean,std,min,25%,50%,75%,max
HematologicStress,13090.0,0.000000e+00,0.506722,-1.608467,-0.324408,-0.043427,0.280992,12.026535
RenalHepaticTurnover,13090.0,-2.605504e-17,0.597050,-1.412255,-0.389773,-0.086600,0.298982,9.631146


## Age-correlation sanity check


In [11]:
for col in axis_train.columns:
    train_corr = np.corrcoef(axis_train[col], train_df[AGE_COL])[0, 1]
    test_corr  = np.corrcoef(axis_test[col],  test_df[AGE_COL])[0, 1]
    print(f"{col}: train corr(age)={train_corr:.3f}, test corr(age)={test_corr:.3f}")


HematologicStress: train corr(age)=-0.000, test corr(age)=0.083
RenalHepaticTurnover: train corr(age)=-0.000, test corr(age)=0.090


## Fit PhysioRisk on axes only

Important: this is the only survival model fit in this notebook.
It uses the physiological axes only, not age/sex, because baseline is already fixed upstream.


In [12]:
def fit_physiorisk_model(axis_train_df, time_col, event_col, penalizer=0.01):
    fit_df = axis_train_df.copy()
    cph = CoxPHFitter(penalizer=penalizer)
    cph.fit(fit_df, duration_col=time_col, event_col=event_col)
    return cph

physio_train_fit = axis_train.copy()
physio_train_fit[TIME_COL] = train_df[TIME_COL].values
physio_train_fit[EVENT_COL] = train_df[EVENT_COL].values

physio_cph = fit_physiorisk_model(
    axis_train_df=physio_train_fit,
    time_col=TIME_COL,
    event_col=EVENT_COL,
    penalizer=PHYSIORISK_PENALIZER,
)

physio_cph.summary


,coef,exp(coef),se(coef),coef lower 95%,coef upper 95%,exp(coef) lower 95%,exp(coef) upper 95%,cmp to,z,p,-log2(p)
covariate,,,,,,,,,,,
HematologicStress,0.432770,1.541521,0.030576,0.372843,0.492697,1.451856,1.636725,0.0,14.154044,1.763185e-45,148.668581
RenalHepaticTurnover,0.235028,1.264944,0.026431,0.183224,0.286832,1.201083,1.332200,0.0,8.892061,5.998564e-19,60.532017


## Score PhysioRisk and combine with frozen BaselineRisk


In [13]:
BENCHMARK_BIOMARKERS = [
    "LBXSAL",
    "LBXSCR",
    "LBXGLU",
    "LBXLYPCT",
    "LBXMCV",
    "LBXRDW",
    "LBXSAPSI",
    "LBXWBCSI",
]

BASE_COLS = [
    ID_COL,
    TIME_COL,
    EVENT_COL,
    AGE_COL,
    SEX_COL,
    BASELINE_COL,
]

OPTIONAL_COLS = ["cycle_start_year"]

KEEP_COLS = [c for c in BASE_COLS + OPTIONAL_COLS + BENCHMARK_BIOMARKERS if c in train_df.columns]



In [14]:
axis_physio_train = pd.Series(
    np.log(physio_cph.predict_partial_hazard(axis_train).values.ravel()),
    index=train_df.index,
    name="PhysioRisk",
)
axis_physio_test = pd.Series(
    np.log(physio_cph.predict_partial_hazard(axis_test).values.ravel()),
    index=test_df.index,
    name="PhysioRisk",
)

phys_mean = float(axis_physio_train.mean())
phys_std = float(axis_physio_train.std())
if phys_std == 0:
    phys_std = 1.0

axis_physio_train = ((axis_physio_train - phys_mean) / phys_std).rename("PhysioRisk")
axis_physio_test  = ((axis_physio_test  - phys_mean) / phys_std).rename("PhysioRisk")

train_scored = train_df[KEEP_COLS].copy()
test_scored  = test_df[KEEP_COLS].copy()

for col in axis_train.columns:
    train_scored[col] = axis_train[col].values
    test_scored[col]  = axis_test[col].values

train_scored["PhysioRisk_2axis"] = axis_physio_train.values
test_scored["PhysioRisk_2axis"]  = axis_physio_test.values

train_scored["TotalRisk_2axis"] = train_scored[BASELINE_COL] + train_scored["PhysioRisk_2axis"]
test_scored["TotalRisk_2axis"]  = test_scored[BASELINE_COL] + test_scored["PhysioRisk_2axis"]

train_scored.head()


,SEQN,PERMTH_INT,MORTSTAT,RIDAGEYR,RIAGENDR,BaselineRisk,cycle_start_year,LBXSAL,LBXSCR,LBXGLU,LBXLYPCT,LBXMCV,LBXRDW,LBXSAPSI,LBXWBCSI,HematologicStress,RenalHepaticTurnover,PhysioRisk_2axis,TotalRisk_2axis
0,2,177.0,1.0,77.0,1.0,2.641589,1999,4.5,0.7,83.7,21.1,88.5,13.7,62.0,7.6,0.229483,-0.564266,-0.118697,2.522892
1,5,244.0,0.0,49.0,1.0,0.055160,1999,4.5,0.8,99.9,37.8,84.9,13.1,63.0,5.9,-0.452632,-0.420010,-1.049939,-0.994780
2,7,237.0,0.0,59.0,2.0,0.467439,1999,4.5,0.6,85.6,46.8,87.4,14.3,75.0,4.9,-0.498282,-0.488210,-1.177476,-0.710037
3,10,231.0,1.0,43.0,1.0,-0.350609,1999,4.2,0.9,89.8,41.3,92.3,13.7,86.0,4.6,-0.231249,0.181944,-0.204271,-0.554880
4,12,236.0,0.0,37.0,1.0,-0.680674,1999,4.7,1.0,82.9,23.7,83.5,13.6,63.0,10.2,0.487152,-0.570170,0.273780,-0.406894


## Evaluation helpers


In [15]:
def c_index_for_score(df, score_col, time_col, event_col):
    return concordance_index(df[time_col], -df[score_col], df[event_col])

def hr_per_1sd(df, score_col, time_col, event_col):
    tmp = df[[time_col, event_col, score_col]].dropna().copy()
    cph = CoxPHFitter()
    cph.fit(tmp, duration_col=time_col, event_col=event_col)
    row = cph.summary.loc[score_col]
    return {
        "HR_per_1SD": float(row["exp(coef)"]),
        "HR_95CI_low": float(row["exp(coef) lower 95%"]),
        "HR_95CI_high": float(row["exp(coef) upper 95%"]),
        "p": float(row["p"]),
    }

def evaluate_scores(df, cohort_name, score_cols, time_col, event_col, age_col):
    rows = []
    for score_col in score_cols:
        hr = hr_per_1sd(df, score_col, time_col, event_col)
        rows.append({
            "cohort": cohort_name,
            "risk_score": score_col,
            "C_index": c_index_for_score(df, score_col, time_col, event_col),
            "HR_per_1SD": hr["HR_per_1SD"],
            "HR_95CI_low": hr["HR_95CI_low"],
            "HR_95CI_high": hr["HR_95CI_high"],
            "p": hr["p"],
            "corr_age": float(np.corrcoef(df[score_col], df[age_col])[0, 1]),
        })
    return pd.DataFrame(rows)


In [16]:
score_cols = [
    "HematologicStress",
    "RenalHepaticTurnover",
    "PhysioRisk_2axis",
    "TotalRisk_2axis",
]

eval_train = evaluate_scores(
    df=train_scored,
    cohort_name="train_temporal",
    score_cols=score_cols,
    time_col=TIME_COL,
    event_col=EVENT_COL,
    age_col=AGE_COL,
)

eval_test = evaluate_scores(
    df=test_scored,
    cohort_name="test_temporal",
    score_cols=score_cols,
    time_col=TIME_COL,
    event_col=EVENT_COL,
    age_col=AGE_COL,
)

eval_all = pd.concat([eval_test, eval_train], ignore_index=True)
eval_all


,cohort,risk_score,C_index,HR_per_1SD,HR_95CI_low,HR_95CI_high,p,corr_age
0,test_temporal,HematologicStress,0.636038,1.668989,1.561461,1.783921,2.368521e-51,8.286705e-02
1,test_temporal,RenalHepaticTurnover,0.604611,2.162374,1.875216,2.493504,2.718387e-26,9.025312e-02
2,test_temporal,PhysioRisk_2axis,0.654110,2.924891,2.796186,3.059520,0.000000e+00,1.062335e-01
3,test_temporal,TotalRisk_2axis,0.839760,1.451989,1.423300,1.481257,1.077953e-293,8.109274e-01
4,train_temporal,HematologicStress,0.586862,1.631751,1.544566,1.723857,2.124257e-68,-1.511312e-16
5,train_temporal,RenalHepaticTurnover,0.558031,1.362211,1.295177,1.432714,3.301666e-33,-4.203927e-17
6,train_temporal,PhysioRisk_2axis,0.602373,1.333612,1.294880,1.373502,1.068879e-81,-1.432347e-16
7,train_temporal,TotalRisk_2axis,0.857525,1.674449,1.652308,1.696887,0.000000e+00,7.903362e-01


## Optional export


In [17]:
from pathlib import Path

EXPORT_DIR = Path("/content/drive/MyDrive/LRE/data/pub")
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

train_output_filepath = EXPORT_DIR / "physiorisk_2axis_v1_train_scored_from_frozen_baseline_levinenocrp.parquet"
test_output_filepath  = EXPORT_DIR / "physiorisk_2axis_v1_test_scored_from_frozen_baseline_levinenocrp.parquet"
eval_output_filepath  = EXPORT_DIR / "physiorisk_2axis_v1_eval_from_frozen_baseline_levinenocrp.csv"

LEVINENOCRP_BIOMARKERS = [
    "LBXSAL",
    "LBXSCR",
    "LBXGLU",
    "LBXLYPCT",
    "LBXMCV",
    "LBXRDW",
    "LBXSAPSI",
    "LBXWBCSI",
]

EXPORT_COLS = [
    ID_COL,
    TIME_COL,
    EVENT_COL,
    AGE_COL,
    SEX_COL,
    "cycle_start_year",
    *LEVINENOCRP_BIOMARKERS,
    "BaselineRisk",
    "HematologicStress",
    "RenalHepaticTurnover",
    "PhysioRisk_2axis",
    "TotalRisk_2axis",
]

missing_export = [c for c in EXPORT_COLS if c not in train_scored.columns]
if missing_export:
    raise ValueError(f"Missing export columns in train_scored: {missing_export}")

train_out = train_scored[EXPORT_COLS].copy()
test_out  = test_scored[EXPORT_COLS].copy()

train_out.to_parquet(train_output_filepath, index=False)
test_out.to_parquet(test_output_filepath, index=False)

# if eval_all exists
if "eval_all" in locals():
    eval_all.to_csv(eval_output_filepath, index=False)

print("Exported:")
print(train_output_filepath)
print(test_output_filepath)
if "eval_all" in locals():
    print(eval_output_filepath)

Exported:
/content/drive/MyDrive/LRE/data/pub/physiorisk_2axis_v1_train_scored_from_frozen_baseline_levinenocrp.parquet
/content/drive/MyDrive/LRE/data/pub/physiorisk_2axis_v1_test_scored_from_frozen_baseline_levinenocrp.parquet
/content/drive/MyDrive/LRE/data/pub/physiorisk_2axis_v1_eval_from_frozen_baseline_levinenocrp.csv


In [18]:
print(train_out.columns.tolist())

['SEQN', 'PERMTH_INT', 'MORTSTAT', 'RIDAGEYR', 'RIAGENDR', 'cycle_start_year', 'LBXSAL', 'LBXSCR', 'LBXGLU', 'LBXLYPCT', 'LBXMCV', 'LBXRDW', 'LBXSAPSI', 'LBXWBCSI', 'BaselineRisk', 'HematologicStress', 'RenalHepaticTurnover', 'PhysioRisk_2axis', 'TotalRisk_2axis']


## Manuscript audit exports

This final section exports manuscript-facing audit artifacts for the locked Levine-noCRP PhysioRisk benchmark pipeline. These cells do not alter the frozen model, preprocessing, residualization, scoring, or evaluation logic. They only write supplementary documentation files needed to make the manuscript reproducible and reviewer-auditable.

The exported files include:

1. A cohort-flow table reporting train/test sample sizes, deaths, and censoring counts.
2. An axis-membership table documenting which residualized biomarkers contribute to each physiological axis and which markers were sign-aligned.
3. A PhysioRisk Cox coefficient table documenting the fitted axis-level physiological model used to generate `PhysioRisk_2axis`.

These exports support the manuscript’s description of DemoRisk, PhysioRisk, and TotalRisk construction without changing any scored outputs.

In [19]:
cohort_flow = pd.DataFrame([
    {
        "cohort": "train_temporal",
        "years": "<=2010",
        "n": len(train_scored),
        "deaths": int(train_scored[EVENT_COL].sum()),
        "censored": int((train_scored[EVENT_COL] == 0).sum()),
    },
    {
        "cohort": "test_temporal",
        "years": ">=2011",
        "n": len(test_scored),
        "deaths": int(test_scored[EVENT_COL].sum()),
        "censored": int((test_scored[EVENT_COL] == 0).sum()),
    },
])
cohort_flow.to_csv(EXPORT_DIR / "physiorisk_2axis_v1_cohort_flow_levinenocrp.csv", index=False)
cohort_flow

,cohort,years,n,deaths,censored
0,train_temporal,<=2010,13090,2520,10570
1,test_temporal,>=2011,6739,416,6323


In [20]:
axis_membership = []

for axis, cols in AXIS_GROUPS_V1.items():
    for col in cols:
        biomarker = col.replace("_resid_z", "")
        axis_membership.append({
            "axis": axis,
            "biomarker": biomarker,
            "residual_column": col,
            "sign_multiplier": SIGN_FLIPS_V1.get(col, 1.0),
            "direction": "higher=worse after sign alignment",
        })

axis_membership = pd.DataFrame(axis_membership)
axis_membership.to_csv(EXPORT_DIR / "physiorisk_2axis_v1_axis_membership.csv", index=False)
axis_membership

,axis,biomarker,residual_column,sign_multiplier,direction
0,HematologicStress,LBXWBCSI,LBXWBCSI_resid_z,1.0,higher=worse after sign alignment
1,HematologicStress,LBXLYPCT,LBXLYPCT_resid_z,-1.0,higher=worse after sign alignment
2,HematologicStress,LBXMCV,LBXMCV_resid_z,1.0,higher=worse after sign alignment
3,HematologicStress,LBXRDW,LBXRDW_resid_z,1.0,higher=worse after sign alignment
4,RenalHepaticTurnover,LBXSCR,LBXSCR_resid_z,1.0,higher=worse after sign alignment
5,RenalHepaticTurnover,LBXSAL,LBXSAL_resid_z,-1.0,higher=worse after sign alignment
6,RenalHepaticTurnover,LBXSAPSI,LBXSAPSI_resid_z,1.0,higher=worse after sign alignment


In [21]:
physio_coef = physio_cph.summary.reset_index()
physio_coef.to_csv(EXPORT_DIR / "physiorisk_2axis_v1_physio_cox_coefficients.csv", index=False)
physio_coef

,covariate,coef,exp(coef),se(coef),coef lower 95%,coef upper 95%,exp(coef) lower 95%,exp(coef) upper 95%,cmp to,z,p,-log2(p)
0,HematologicStress,0.432770,1.541521,0.030576,0.372843,0.492697,1.451856,1.636725,0.0,14.154044,1.763185e-45,148.668581
1,RenalHepaticTurnover,0.235028,1.264944,0.026431,0.183224,0.286832,1.201083,1.332200,0.0,8.892061,5.998564e-19,60.532017
